# 1 hidden class - encoder generator

## Import libraries

In [1]:
import torch
from torch import nn, optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import davies_bouldin_score
from sklearn.decomposition import PCA
from scipy.stats import chi2
from sklearn.mixture import GaussianMixture
from collections import deque, defaultdict

# Import training, validation and test data

In [2]:
train = pd.read_csv('../train.csv')
val = pd.read_csv('../val.csv')
test = pd.read_csv('../test.csv')

'Infilteration' and 'DoS attacks-SlowHTTPTest' classes were aparently mislabeled, so we will exclude them, remaining 12 attack classes

In [3]:
classes_out = ['Infilteration','DoS attacks-SlowHTTPTest']
train = train.drop(index=train[train['Label'].isin(classes_out)].index)
val = val.drop(index=val[val['Label'].isin(classes_out)].index)
test = test.drop(index=test[test['Label'].isin(classes_out)].index)

11 attack classes will be joined into major attack classes
* FTP-BruteForce and SSH-Bruteforce -> BruteForce
* DoS attacks-GoldenEye, DoS attacks-Slowloris and DoS attacks-Hulk -> DoS
* DDoS attacks-LOIC-HTTP, DDOS attack-LOIC-UDP and DDOS attack-HOIC -> DDoS
* Brute Force -Web, Brute Force -XSS and SQL Injection -> DDoS

Resulting in 5 major attack classes: BruteForce, DoS, DDoS, Web and Bot, and Benign class 

In [4]:
train['Label'] = train['Label'].replace('FTP-BruteForce','BruteForce')
train['Label'] = train['Label'].replace('SSH-Bruteforce','BruteForce')
train['Label'] = train['Label'].replace('DoS attacks-GoldenEye','DoS')
train['Label'] = train['Label'].replace('DoS attacks-Slowloris','DoS')
train['Label'] = train['Label'].replace('DoS attacks-Hulk','DoS')
train['Label'] = train['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
train['Label'] = train['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
train['Label'] = train['Label'].replace('DDOS attack-HOIC','DDoS')
train['Label'] = train['Label'].replace('Brute Force -Web','Web')
train['Label'] = train['Label'].replace('Brute Force -XSS','Web')
train['Label'] = train['Label'].replace('SQL Injection','Web')


val['Label'] = val['Label'].replace('FTP-BruteForce','BruteForce')
val['Label'] = val['Label'].replace('SSH-Bruteforce','BruteForce')
val['Label'] = val['Label'].replace('DoS attacks-GoldenEye','DoS')
val['Label'] = val['Label'].replace('DoS attacks-Slowloris','DoS')
val['Label'] = val['Label'].replace('DoS attacks-Hulk','DoS')
val['Label'] = val['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
val['Label'] = val['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
val['Label'] = val['Label'].replace('DDOS attack-HOIC','DDoS')
val['Label'] = val['Label'].replace('Brute Force -Web','Web')
val['Label'] = val['Label'].replace('Brute Force -XSS','Web')
val['Label'] = val['Label'].replace('SQL Injection','Web')


test['Label'] = test['Label'].replace('FTP-BruteForce','BruteForce')
test['Label'] = test['Label'].replace('SSH-Bruteforce','BruteForce')
test['Label'] = test['Label'].replace('DoS attacks-GoldenEye','DoS')
test['Label'] = test['Label'].replace('DoS attacks-Slowloris','DoS')
test['Label'] = test['Label'].replace('DoS attacks-Hulk','DoS')
test['Label'] = test['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
test['Label'] = test['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
test['Label'] = test['Label'].replace('DDOS attack-HOIC','DDoS')
test['Label'] = test['Label'].replace('Brute Force -Web','Web')
test['Label'] = test['Label'].replace('Brute Force -XSS','Web')
test['Label'] = test['Label'].replace('SQL Injection','Web')

## Labels into numeric

In [5]:
labels_str = [i for i in train['Label'].value_counts().index]
print(labels_str)

for i, l in enumerate(labels_str):
    train['Label'] = train['Label'].replace(l, i)
    val['Label'] = val['Label'].replace(l, i)
    test['Label'] = test['Label'].replace(l, i)

['DDoS', 'Benign', 'DoS', 'BruteForce', 'Bot', 'Web']


C:\Users\Lincoln\AppData\Local\Temp\ipykernel_8000\2307520911.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train['Label'] = train['Label'].replace(l, i)
C:\Users\Lincoln\AppData\Local\Temp\ipykernel_8000\2307520911.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  val['Label'] = val['Label'].replace(l, i)
C:\Users\Lincoln\AppData\Local\Temp\ipykernel_8000\2307520911.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `res

## Contrastive AutoEncoder Functions

<img src="..\img\contrastive_pairs.png" alt="Contrastive Pairs" width="60%"/>

In [6]:
def pick_pair(embeddings, labels, pick_embeddings, pick_labels):
    """
    Generates all pairs between embeddings (batch) and pick_embeddings (reference samples), with labels_pair indicating whether the pairs are positive (0) or negative (1). 
    """
    N, D = embeddings.shape
    M = pick_embeddings.shape[0]

    # Expands to all possible combinations
    A = embeddings.unsqueeze(0).expand(M, N, D).reshape(M * N, D)
    B = pick_embeddings.unsqueeze(1).expand(M, N, D).reshape(M * N, D)

    labels_exp = labels.unsqueeze(0).expand(M, N)
    pick_labels_exp = pick_labels.unsqueeze(1).expand(M, N)
    labels_pair = (labels_exp != pick_labels_exp).reshape(-1).float()

    return A, B, labels_pair

def ContrastiveLoss(input1, input2, y, margin=1.0):
    # y = 0 when input1 and input2 belong to the same class, and y = 1 otherwise.
    diff = input1 - input2
    dist_sq = torch.sum(torch.pow(diff,2),1)
    dist = torch.sqrt(dist_sq + 1e-10) # "Adds 1e-10 because if the sqrt is 0, the gradient becomes NaN.
    mdist = margin - dist
    mdist = F.relu(mdist)
    loss = ((1-y) * dist * dist) + (y * mdist * mdist)
    return torch.mean(loss)

def AEContrastive(input, output, embeddings, labels, pick_encoded, pick_labels, margin=1.0, lambda1=1.0):
    mseloss = nn.MSELoss()
    ae_loss = mseloss(input,output)

    A, B, labels_pair = pick_pair(embeddings, labels, pick_encoded, pick_labels)
    contrastive_loss = ContrastiveLoss(A, B, labels_pair, margin=margin)
    return ae_loss + contrastive_loss * lambda1, ae_loss, contrastive_loss

In [7]:
class contrastive_ae(nn.Module):
    def __init__(self, input_neurons, neurons):
        super().__init__()
        self.encoder0 = nn.Linear(input_neurons,64)
        self.encoder1 = nn.Linear(64,32)
        self.encoder2 = nn.Linear(32,16)
        self.encoder3 = nn.Linear(16,neurons)

        self.decoder0 = nn.Linear(neurons,16)
        self.decoder1 = nn.Linear(16,32)
        self.decoder2 = nn.Linear(32,64)
        self.decoder3 = nn.Linear(64,input_neurons)

        self.activation0 = nn.ReLU()
    
    def forward(self, X):
        X = self.activation0(self.encoder0(X))
        X = self.activation0(self.encoder1(X))
        X = self.activation0(self.encoder2(X))
        X = self.encoder3(X) # No activation function
        encoded = X
        X = self.activation0(self.decoder0(X))
        X = self.activation0(self.decoder1(X))
        X = self.activation0(self.decoder2(X))
        X = self.decoder3(X) # No activation function


        return X, encoded

In [8]:
class encoder(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.encoder0 = list(model.children())[0]
        self.encoder1 = list(model.children())[1]
        self.encoder2 = list(model.children())[2]
        self.encoder3 = list(model.children())[3]
        self.activation0 = list(model.children())[8]
    
    def forward(self, X):
        X = self.activation0(self.encoder0(X))
        X = self.activation0(self.encoder1(X))
        X = self.activation0(self.encoder2(X))
        X = self.encoder3(X)
        return X

In [9]:
from torch.utils.data import Dataset, DataLoader,TensorDataset
import random

# Prepare reference samples for each batch

def build_class_reference_batches(X_train, y_train, batch_size=512):
    classes = torch.unique(y_train).tolist()
    n_classes = len(classes)
    total_samples = len(X_train)
    dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    

    print(classes)

    # Maps each class to its indices
    class_to_indices = {
        int(cls.item()): (y_train == cls).nonzero(as_tuple=True)[0].tolist()
        for cls in torch.unique(y_train)
    }

    reference_list = []
    total_batches = len(train_loader)

    for batch_idx in range(total_batches):
        start = batch_idx * batch_size
        end = min(start + batch_size, total_samples)
        batch_indices = set(range(start, end))

        class_samples = []
        for cls in classes:
            candidates = [i for i in class_to_indices[cls] if i not in batch_indices]
            if not candidates:
                raise ValueError(f"Class {cls} has no samples outside of batch {batch_idx}")
            chosen = candidates[torch.randint(len(candidates), (1,)).item()]
            class_samples.append(chosen)

        reference_list.append(class_samples)


    return train_loader, reference_list

In [10]:
# Normalization function

def normalize(t):
    t_norm = (t - t.mean()) / t.std()
    return t_norm


In [ ]:
array_hidden_classes = [[0],[2],[3],[4],[5]] # Class 1 correspond to benign class
filenames = ['0','2','3','4','5']

total_classes = len(labels_str)
for i in range(len(array_hidden_classes)):
    filename = f'{filenames[i]}_hidden'
    print(filename)
    hidden_classes = array_hidden_classes[i] # Classes hidden from training

    hidden_classes_index = list(train[train['Label'].isin(hidden_classes)].index)

    train_hidden = train.loc[hidden_classes_index]

    train_not_hidden = train.drop(hidden_classes_index)

    X_train = train_not_hidden.drop(columns=['Label'])
    y_train = train_not_hidden['Label']
    X_val = val.drop(columns=['Label'])
    y_val = val['Label'].values
    X_test = test.drop(columns=['Label'])
    y_test = test['Label'].values

    torch.manual_seed(123)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(device)
    X_train = torch.tensor(np.array(X_train), dtype=torch.float)
    y_train = torch.tensor(np.array(y_train), dtype=torch.float)

    train_loader, pick_list = build_class_reference_batches(X_train,y_train, batch_size=512)
    pick_samples = []
    pick_labels = []
    for l in pick_list:
        pick_samples.append(X_train[l])
        pick_labels.append(y_train[l])

    pick_samples = torch.stack(pick_samples)    
    pick_labels = torch.stack(pick_labels)


    neurons = total_classes - len(hidden_classes)
    print(f'{X_train.shape[1]} {neurons}')
    model = contrastive_ae(input_neurons=X_train.shape[1],neurons=neurons)
    model.to(device)

    learning_rate = 0.0001
    opt = optim.Adam(model.parameters(),lr=learning_rate)
    margin = 10 # Class separation parameter
    cae_lambda_1 = 1.0 # Contrastive learning weight 
    
    
    for epoch in range(250):
        running_loss = 0
        running_ael = 0
        running_cl = 0
        counter = 0
        for data in train_loader:
            inputs, labels = data
            inputs = inputs.to(device)
            labels = labels.to(device)
            ps = pick_samples[counter]
            pl = pick_labels[counter]
            ps = ps.to(device)
            pl = pl.to(device)
            counter += 1
            opt.zero_grad()
            outputs, encoded = model(inputs)
            _, pick_encoded = model(ps)
            tam_pe = pick_encoded.shape[0]
            enc_total = torch.cat((encoded,pick_encoded), dim=0)
            enc_total_norm = normalize(enc_total)
            encoded_norm = enc_total_norm[:-tam_pe]
            pick_encoded_norm = enc_total_norm[-tam_pe:]
            loss, ael, cl = AEContrastive(input=inputs, output=outputs, embeddings = encoded_norm, labels=labels, pick_encoded = pick_encoded_norm, pick_labels = pl, margin=margin, lambda1=cae_lambda_1)
            loss.backward()
            opt.step()
            running_loss += loss.item()
            running_ael += ael.item()
            running_cl += cl.item()
        print(f'filename: {filename} Epoch {epoch+1} loss:{running_loss/len(train_loader)} ael:{running_ael/len(train_loader)} cl:{running_cl/len(train_loader)}')

    model_e = encoder(model=model)
    model_e.to(device)

    model_e.eval()

    X_train = torch.tensor(np.array(X_train), dtype=torch.float)
    y_train = torch.tensor(np.array(y_train), dtype=torch.float)

    train_encoded = model_e(X_train.to(device))

    train_encoded = train_encoded.detach().cpu().numpy()

    train_encoded_df = pd.DataFrame(train_encoded)
    train_encoded_df['Label'] = y_train.detach().cpu().numpy().astype(int)
    display(train_encoded_df)

    X_val = torch.tensor(np.array(X_val), dtype=torch.float)

    val_encoded = model_e(X_val.to(device))

    val_encoded = val_encoded.detach().cpu().numpy()

    val_encoded_df = pd.DataFrame(val_encoded)

    val_encoded_df['Label'] = y_val

    display(val_encoded_df)

    X_test = torch.tensor(np.array(X_test), dtype=torch.float)

    test_encoded = model_e(X_test.to(device))

    test_encoded = test_encoded.detach().cpu().numpy()

    test_encoded_df = pd.DataFrame(test_encoded)

    test_encoded_df['Label'] = y_test

    display(test_encoded_df)

    train_encoded_df.to_csv(f'train_encoded_{filename}.csv',index=False)
    val_encoded_df.to_csv(f'val_encoded_{filename}.csv',index=False)
    test_encoded_df.to_csv(f'test_encoded_{filename}.csv',index=False)